# 🧠 LECO Auto-Recloser Failure Prediction: Deep Learning (Google Colab)

The same question as `AR_ML_Colab.ipynb`, answered by a neural network instead of LightGBM:

> **Will this recloser fail within the next 30 days?**

**What is different?**
* LightGBM looks at one hour at a time, plus a few hand-made 7-day trends.
* The network reads the **last 14 days of each recloser as a sequence** and learns the warning patterns over
  time by itself.
* One network answers three questions at once: failure within 30 days, failure within 7 days, and which part
  will fail.

**A fair comparison.** This notebook loads the data, splits the months, calibrates and chooses the alarm level
with **exactly the same code** as the LightGBM notebook. If `ar_ml_model.zip` (the LightGBM result) is in the
same Drive folder, both models are scored on the same test months, next to the Health Index.

**How to run it** (details in `docs/ML_COLAB_GUIDE.md`):
1. Use the same data as before: `ar_ml_dataset_hourly.csv.gz` in the Google Drive folder **AR_ML**.
   Optionally, put `ar_ml_model.zip` next to it.
2. In Colab, choose **File → Upload notebook** and pick this file.
3. Choose **Runtime → Change runtime type → T4 GPU** (the free GPU is enough), then **Runtime → Run all**.
4. At the end, the browser downloads **`ar_dl_model.zip`**. A copy is also saved in Drive/AR_ML.
5. On the PC, unzip it into **`models/dl/`** and run `python main.py dl-score`. The dashboard's **Deep Learning**
   tab then shows the network's risk.

| Section | What happens |
|---|---|
| 0 | Settings, packages, GPU check |
| 1–4 | Same as the LightGBM notebook: load the data, list the failures, build the inputs, split by time |
| 5 | Turn the hourly rows into 14-day sequences |
| 6 | Three networks: GRU, TCN and Transformer |
| 7 | Train each network; keep the best one on the validation months |
| 8–9 | Turn scores into realistic probabilities; choose the alarm level |
| 10 | Honest test: deep learning vs LightGBM vs Health Index |
| 11 | One recloser over time |
| 12 | Which part is likely to fail |
| 13 | What the network looks at (gradients) |
| 14 | Daily HI vs deep-learning agreement |
| 15 | Export `ar_dl_model.zip` |

## 0. Settings and packages

In [ ]:
import os
SMOKE = os.environ.get("AR_NB_SMOKE") == "1"   # True = tiny run that only checks the notebook works
DATA_FILE = "ar_ml_dataset_hourly.csv.gz"     # made on the PC by: python main.py generate-ml-data
DRIVE_FOLDER = "AR_ML"                        # Google Drive folder that holds DATA_FILE
LGBM_ZIP = "ar_ml_model.zip"                  # optional: LightGBM result of AR_ML_Colab.ipynb, for the comparison
HORIZON_DAYS = 30                             # the question: failure within the next 30 days?
WINDOW_STEPS = 168                            # steps of history the network reads for one prediction
STEP_HOURS = 2                                # hours between steps: 168 x 2 h = the last 14 days
ARCHS = ["gru", "tcn", "transformer"]         # networks to try; the best on the validation months is kept
HIDDEN = 16 if SMOKE else 64                  # network width
EPOCHS = 2 if SMOKE else 15                   # training rounds, each on a fresh sample of windows
PATIENCE = 4                                  # stop after this many rounds without a better validation score
SAMPLES_PER_EPOCH = 4_096 if SMOKE else 409_600
BATCH = 256 if SMOKE else 1024
POS_SHARE = 0.3                               # share of "fails within 30 days" windows in each round
LEARNING_RATE = 2e-3
FALSE_ALARM_BUDGET = 1.0                      # max false-alarm days per recloser per month for the alarm level
HI_HEALTHY_MIN = 60                           # Phase 1 rule: HI >= 60 means healthy
SEED = 42

In [ ]:
import importlib, subprocess, sys
try:
    importlib.import_module("lightgbm")        # only used to score the LightGBM bundle for the comparison
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])

import datetime as dt, json, shutil, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.3})
IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"                   # mixed precision: about twice as fast on a GPU
torch.manual_seed(SEED)
gpu = torch.cuda.get_device_name(0) if AMP else "no GPU"
print("torch", torch.__version__, "|", DEVICE, f"({gpu})", "| Colab:", IN_COLAB, "| SMOKE:", SMOKE)
if DEVICE.type == "cpu" and not SMOKE:
    print("WARNING: no GPU. Choose Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all. "
          "On a CPU the training takes many hours.")

## 1. Load the data
Same as the LightGBM notebook. The cell first looks in Google Drive (`MyDrive/AR_ML/`) and asks for permission to
mount it. If the file is not there, an upload box appears instead.

In [ ]:
candidates = [os.environ.get("AR_ML_DATA", ""), DATA_FILE, f"/content/drive/MyDrive/{DRIVE_FOLDER}/{DATA_FILE}",
              f"/content/drive/MyDrive/{DATA_FILE}", f"/content/{DATA_FILE}",
              os.path.join("..", "output", "ml_generated", DATA_FILE)]
path = next((p for p in candidates if p and os.path.exists(p)), None)
if path is None and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    path = next((p for p in candidates if p and os.path.exists(p)), None)
    if path is None:
        from google.colab import files
        print(f"{DATA_FILE} is not in Drive/{DRIVE_FOLDER} - choose it from your computer instead")
        path = next(iter(files.upload()))
if path is None:
    raise FileNotFoundError(f"Put {DATA_FILE} in Google Drive/{DRIVE_FOLDER}, next to this notebook, or set AR_ML_DATA")
print("Using", path)

In [ ]:
t0 = time.time()
BOOL_COLS = ["telemetry_present", "is_healthy", "failed_now", "failure_next_7d", "failure_next_30d",
             "censored_7d", "censored_30d"]
parts, keep_ids = [], None
for chunk in pd.read_csv(path, chunksize=300_000, parse_dates=["ts"], low_memory=False):
    chunk = chunk[chunk["ts"].dt.minute == 0]                  # one row per recloser per hour
    if SMOKE:                                                  # tiny run: only 15 reclosers
        keep_ids = keep_ids or sorted(chunk["asset_id"].unique())[:15]
        chunk = chunk[chunk["asset_id"].isin(keep_ids)]
    floats = chunk.select_dtypes("float64").columns.drop("days_to_next_failure", errors="ignore")
    chunk[floats] = chunk[floats].astype("float32")            # halves the memory
    parts.append(chunk)
df = pd.concat(parts, ignore_index=True).sort_values(["asset_id", "ts"]).reset_index(drop=True)
del parts
for c in BOOL_COLS:
    df[c] = df[c].astype(str).str.lower().eq("true")
print(f"{len(df):,} hourly rows | {df['asset_id'].nunique()} reclosers | "
      f"{df['ts'].min():%Y-%m-%d} -> {df['ts'].max():%Y-%m-%d} (UTC) | loaded in {time.time() - t0:.0f} s")

## 2. The failures
Each row carries the days until that recloser's next recorded failure. The list of failures is rebuilt from it.

In [ ]:
f = df.loc[df["days_to_next_failure"].notna(), ["asset_id", "ts", "days_to_next_failure", "next_failure_component"]].copy()
f["failure_time"] = f["ts"] + pd.to_timedelta(f["days_to_next_failure"].astype(float), unit="D")
f = f.sort_values(["asset_id", "failure_time"])
new = (f["asset_id"] != f["asset_id"].shift()) | (f["failure_time"].diff() > pd.Timedelta(hours=2))
failures = (f.loc[new, ["asset_id", "failure_time", "next_failure_component"]]
            .rename(columns={"next_failure_component": "component"}).reset_index(drop=True))
failures["failure_time"] = failures["failure_time"].dt.round("min")
print(f"{len(failures)} failures in {failures['asset_id'].nunique()} reclosers | "
      f"rows labelled 'fails within 30 days': {df['failure_next_30d'].mean():.1%}")
display(failures["component"].value_counts().rename_axis("component").to_frame("failures"))

## 3. Model inputs (features)
The **same feature recipe the dashboard uses** (`ar_gen/ml_features.py`, copied in automatically): V1–V41,
weather / recloser model / comms channel as yes-no columns, and 7-day trends of 13 key variables. The HI and the
label columns are not inputs. Here these inputs form **one step** of the sequence the network reads.

In [ ]:
%%writefile ar_ml_features.py
"""ML feature recipe shared by the Google Colab notebook and the live scorer.

KEEP THIS FILE SELF-CONTAINED (only numpy and pandas): the Colab notebook embeds it verbatim,
so training and live scoring build exactly the same features.

Input: V1-V42 rows (one per recloser per interval, every interval present) with the columns
asset_id, ts, telemetry_present, v01..v42, model, comms_channel.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

FEATURE_VERSION = 1

V_FEATURES = [
    "v01_battery_voltage_v", "v02_discharge_episodes_24h", "v03_soh_est_pct", "v04_battery_life_used_pct",
    "v05_ac_failures_30d", "v06_max_dod_30d_pct", "v07_battery_alarms_30d", "v08_battery_temp_age_years",
    "v09_auto_test_failed_count", "v10_minutes_on_battery_30d", "v11_wear_max_pct", "v12_lightning_faults_30d",
    "v13_permanent_faults_30d", "v14_temporary_faults_30d", "v15_earth_faults_30d", "v16_sef_faults_30d",
    "v17_oc_faults_30d", "v18_max_fault_current_30d_a", "v19_faults_24h", "v20_lockouts_30d", "v21_co_total",
    "v22_open_ops_30d", "v23_close_ops_30d", "v24_protection_trips_30d", "v25_auto_reclose_ops_30d",
    "v26_switching_ops_24h", "v27_unsuccessful_ops_30d", "v28_scada_availability_24h_pct",
    "v29_comm_failures_24h", "v30_modem_restarts_30d", "v31_signal_strength_24h_dbm", "v32_reconnects_24h",
    "v33_module_faults_30d", "v34_diag_warnings_30d", "v35_firmware_versions_behind", "v36_rs485_errors_30d",
    "v37_sg_age_years", "v38_years_in_service", "v39_days_since_maintenance", "v40_ambient_temp_c",
    "v41_coastal_flag",
]

# Variables whose 7-day change and 7-day mean are added: failures show up as things getting worse.
TREND_VARS = [
    "v01_battery_voltage_v", "v03_soh_est_pct", "v06_max_dod_30d_pct", "v07_battery_alarms_30d",
    "v10_minutes_on_battery_30d", "v11_wear_max_pct", "v27_unsuccessful_ops_30d",
    "v28_scada_availability_24h_pct", "v29_comm_failures_24h", "v30_modem_restarts_30d",
    "v31_signal_strength_24h_dbm", "v33_module_faults_30d", "v34_diag_warnings_30d",
]
TREND_HOURS = 168

# Fixed category lists -> the one-hot columns never change between training and scoring.
CATEGORIES = {
    "v42_weather": ["SUNNY", "CLOUDY", "RAINY", "STORM"],
    "model": ["OSM15-310", "OSM27-310"],
    "comms_channel": ["MOBILE", "FIBRE"],
}


def feature_names() -> list[str]:
    names = ["telemetry_present", *V_FEATURES]
    names += [f"{v}_{kind}" for v in TREND_VARS for kind in ("chg7d", "mean7d")]
    names += [f"{col}__{cat}" for col, cats in CATEGORIES.items() for cat in cats]
    return names


def build_features(df: pd.DataFrame, step_minutes: int) -> pd.DataFrame:
    """Model inputs for every row of ``df`` (same row order and index).

    step_minutes: spacing of the rows of one recloser (60 for the hourly training file,
    15 for live data). The 7-day trend windows are defined in time, so both agree.
    """
    rows = max(1, int(round(TREND_HOURS * 60 / step_minutes)))
    order = df.sort_values(["asset_id", "ts"], kind="stable").index
    d = df.loc[order]
    out = pd.DataFrame(index=d.index)
    present = d["telemetry_present"]
    if present.dtype == object:
        present = present.astype(str).str.lower().isin(["true", "1"])
    out["telemetry_present"] = present.astype(float)
    for c in V_FEATURES:
        out[c] = pd.to_numeric(d[c], errors="coerce").astype(float)
    groups = d["asset_id"]
    for v in TREND_VARS:
        x = out[v]
        out[f"{v}_chg7d"] = x - x.groupby(groups).shift(rows)
        out[f"{v}_mean7d"] = x.groupby(groups).transform(lambda s: s.rolling(rows, min_periods=1).mean())
    for col, cats in CATEGORIES.items():
        values = d[col].astype(str) if col in d.columns else pd.Series("", index=d.index)
        for cat in cats:
            out[f"{col}__{cat}"] = (values == cat).astype(float)
    out = out[feature_names()]
    return out.loc[df.index].astype(np.float64)

In [ ]:
import ar_ml_features
importlib.reload(ar_ml_features)
from ar_ml_features import FEATURE_VERSION, build_features, feature_names

t0 = time.time()
X_all = build_features(df, step_minutes=60).astype("float32")
y_all = df["failure_next_30d"].astype(int).to_numpy()
FEATURES = feature_names()
print(f"feature version {FEATURE_VERSION}: {len(FEATURES)} inputs for {len(X_all):,} rows in {time.time() - t0:.0f} s")

## 4. Split by time, never randomly
Exactly the same split as the LightGBM notebook: **fit** (oldest data, trains the network), **valid** (the next
3 months: picks the best network and epoch, calibration and the alarm level) and **test** (the newest ~35 %, the
honest exam), with a 30-day gap between the parts. Rows whose outcome is unknown, or where the recloser is
already broken, are left out as prediction points. They still appear inside the 14-day histories.

In [ ]:
def safe_ap(y, s):
    return average_precision_score(y, s) if 0 < np.sum(y) < len(y) else float("nan")

def safe_auc(y, s):
    return roc_auc_score(y, s) if 0 < np.sum(y) < len(y) else float("nan")

usable = (~df["censored_30d"] & ~df["failed_now"]).to_numpy()
t_first, t_last = df.loc[usable, "ts"].min(), df.loc[usable, "ts"].max()
GAP = pd.Timedelta(days=HORIZON_DAYS)
TEST_START = (t_first + (t_last - t_first) * 0.65).normalize()
VALID_START = TEST_START - GAP - pd.Timedelta(days=90)
ts = df["ts"]
fit_mask = usable & (ts < VALID_START - GAP).to_numpy()
valid_mask = usable & ((ts >= VALID_START) & (ts < TEST_START - GAP)).to_numpy()
test_mask = usable & (ts >= TEST_START).to_numpy()

def failures_between(lo, hi):
    return failures[(failures["failure_time"] > lo) & (failures["failure_time"] <= hi)]

split = pd.DataFrame({
    "from": [t_first, VALID_START, TEST_START], "to": [VALID_START - GAP, TEST_START - GAP, t_last],
    "rows": [fit_mask.sum(), valid_mask.sum(), test_mask.sum()],
    "positive rows %": [100 * y_all[m].mean() for m in (fit_mask, valid_mask, test_mask)],
    "failures": [len(failures_between(t_first, VALID_START)), len(failures_between(VALID_START, TEST_START)),
                 len(failures_between(TEST_START, t_last + GAP))]}, index=["fit", "valid", "test"])
display(split.round(1))
for part, m in (("fit", fit_mask), ("valid", valid_mask), ("test", test_mask)):
    if y_all[m].sum() == 0:
        raise RuntimeError(f"The {part} months contain no failures, so the model cannot learn or be checked. "
                           "Generate more data on the PC (more reclosers or more days) and run again.")

## 5. From table rows to sequences
For every prediction, the network reads **168 snapshots, 2 hours apart: the last 14 days** of that recloser.
* **One scale.** Counts with a long tail (operations, fault current, minutes on battery …) are log-compressed.
  Then every input is standardised with averages from the **fit months only**.
* **Missing values** become 0, which is the average. `telemetry_present` still tells the network when data was missing.
* **Short history.** For a recloser's first 14 days, the window is padded with its first hour.
* **Memory.** Windows are cut on the fly from one array on the GPU, so memory stays small.

The next cell writes the **project's deep-learning code** (`ar_gen/dl_model.py`, copied in automatically): the
networks, the input scaling and the window rule. The dashboard rebuilds the trained network with the same file.

In [ ]:
%%writefile ar_dl_model.py
"""Deep-learning network shared by the Colab notebook (AR_DL_Colab.ipynb) and the live scorer.

KEEP THIS FILE SELF-CONTAINED (only numpy and torch): the notebook embeds it verbatim, so the
network trained in Colab is rebuilt here, from its weights, with exactly the same code.

A prediction reads one window: the last ``steps`` rows of a recloser, ``step_hours`` apart, each row
holding the inputs of ml_features.build_features after scale_inputs.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

NET_VERSION = 1          # change when ARNet changes so older exported weights are refused, not misread
ARCHS = ("gru", "tcn", "transformer")


class TCNBlock(nn.Module):
    """Two causal, dilated 1-D convolutions with a shortcut."""

    def __init__(self, width, dilation, dropout):
        super().__init__()
        self.pad = 2 * dilation
        self.conv1 = nn.Conv1d(width, width, 3, dilation=dilation)
        self.conv2 = nn.Conv1d(width, width, 3, dilation=dilation)
        self.norm = nn.BatchNorm1d(width)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):                                       # batch x width x steps
        y = self.drop(F.gelu(self.conv1(F.pad(x, (self.pad, 0)))))
        y = self.drop(self.conv2(F.pad(y, (self.pad, 0))))
        return F.gelu(self.norm(x + y))


class ARNet(nn.Module):
    """Sequence encoder (GRU, TCN or Transformer) + attention pooling + three heads:
    failure within 30 days, failure within 7 days (logits) and which part fails (logits per part)."""

    def __init__(self, arch, n_inputs, n_components, width=64, steps=168, dropout=0.2):
        super().__init__()
        self.arch = arch
        if arch == "gru":
            self.inp = nn.Conv1d(n_inputs, width, 5, padding=2)
            self.gru = nn.GRU(width, width, num_layers=2, batch_first=True, dropout=dropout)
        elif arch == "tcn":
            self.inp = nn.Conv1d(n_inputs, width, 1)
            self.blocks = nn.ModuleList(TCNBlock(width, 2 ** k, dropout) for k in range(7))  # reaches 509 steps back
        elif arch == "transformer":
            self.inp = nn.Linear(n_inputs, width)
            self.pos = nn.Parameter(0.02 * torch.randn(1, steps, width))
            layer = nn.TransformerEncoderLayer(width, nhead=4, dim_feedforward=2 * width, dropout=dropout,
                                               activation="gelu", batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        else:
            raise ValueError(f"unknown network {arch!r}: use one of {', '.join(ARCHS)}")
        self.attn = nn.Linear(width, 1)
        self.mix = nn.Sequential(nn.LayerNorm(2 * width), nn.Linear(2 * width, width), nn.GELU(), nn.Dropout(dropout))
        self.head30 = nn.Linear(width, 1)
        self.head7 = nn.Linear(width, 1)
        self.head_part = nn.Linear(width, max(1, n_components))

    def encode(self, x):                                        # batch x steps x inputs -> batch x steps x width
        if self.arch == "gru":
            return self.gru(F.gelu(self.inp(x.transpose(1, 2))).transpose(1, 2))[0]
        if self.arch == "tcn":
            h = self.inp(x.transpose(1, 2))
            for block in self.blocks:
                h = block(h)
            return h.transpose(1, 2)
        return self.enc(self.inp(x) + self.pos[:, -x.shape[1]:])

    def forward(self, x):
        h = self.encode(x)
        w = torch.softmax(self.attn(h).squeeze(-1).float(), dim=1)          # how much each step matters
        pooled = (w.unsqueeze(-1) * h.float()).sum(dim=1).to(h.dtype)
        z = self.mix(torch.cat([h[:, -1], pooled], dim=-1))                 # latest step + weighted summary
        return self.head30(z).squeeze(-1), self.head7(z).squeeze(-1), self.head_part(z)


def scale_inputs(raw, log_idx, mean, std, clip):
    """Raw inputs (rows x features, from build_features) -> what the network reads.

    Long-tailed counts (columns log_idx) are log-compressed, then every input is standardised with the
    fit-month mean/std. Missing values become 0 (the average); values are limited to +-clip.
    """
    v = np.array(raw, dtype=np.float32)
    v[:, log_idx] = np.log1p(np.clip(v[:, log_idx], 0, None))
    v -= np.asarray(mean, dtype=np.float32)
    v /= np.asarray(std, dtype=np.float32)
    np.nan_to_num(v, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(v, -clip, clip, out=v)


def window_rows(end_rows, first_rows, steps, stride):
    """Row numbers of each window, oldest first: ``steps`` rows ``stride`` apart that end at ``end_rows``.

    Rows are one recloser's regular time series; positions before its first row (``first_rows``) repeat
    that first row, as for a recloser with less history than the window.
    """
    offsets = np.arange(-(steps - 1) * stride, 1, stride)
    return np.maximum(np.asarray(end_rows)[:, None] + offsets[None, :], np.asarray(first_rows)[:, None])

In [ ]:
import ar_dl_model
importlib.reload(ar_dl_model)
from ar_dl_model import NET_VERSION, ARNet, scale_inputs, window_rows

t0 = time.time()
CLIP = 8.0                                     # scaled inputs are limited to +-8 standard deviations
rng = np.random.default_rng(SEED)
fit_idx = np.flatnonzero(fit_mask)
stat = pd.DataFrame(X_all.iloc[np.sort(rng.choice(fit_idx, min(len(fit_idx), 500_000), replace=False))]
                    .to_numpy(np.float32), columns=FEATURES)
LOG_COLS = [c for c in FEATURES if stat[c].min() >= 0 and stat[c].quantile(0.99) > 50]
LOG_IDX = [FEATURES.index(c) for c in LOG_COLS]
stat[LOG_COLS] = np.log1p(stat[LOG_COLS])
MU = stat.mean().fillna(0).to_numpy(np.float32)
SD = stat.std().to_numpy(np.float32)
SD = np.where(np.isfinite(SD) & (SD > 1e-6), SD, 1.0).astype(np.float32)
del stat

X_seq = np.empty((len(X_all), len(FEATURES)), dtype=np.float32)
for i in range(0, len(X_all), 500_000):
    X_seq[i:i + 500_000] = scale_inputs(X_all.iloc[i:i + 500_000].to_numpy(np.float32), LOG_IDX, MU, SD, CLIP)

# Rows are sorted by recloser and hour: a window is "this row and every STEP_HOURS-th row before it".
ids = df["asset_id"].to_numpy()
first = np.r_[0, np.flatnonzero(ids[1:] != ids[:-1]) + 1]
ROW_START_NP = np.repeat(first, np.diff(np.r_[first, len(ids)]))
hours = df["ts"].diff().dt.total_seconds().div(3600).to_numpy()
odd = np.count_nonzero((hours != 1) & (np.arange(len(ids)) != ROW_START_NP))
if odd:
    print(f"WARNING: {odd} rows are not exactly 1 hour after the previous row; windows assume hourly rows")

X_dev = torch.from_numpy(X_seq).to(DEVICE)
del X_seq
ROW_START = torch.from_numpy(ROW_START_NP).to(DEVICE)
OFFSETS = torch.arange(-(WINDOW_STEPS - 1) * STEP_HOURS, 1, STEP_HOURS, device=DEVICE)

def windows(rows):
    # (batch, WINDOW_STEPS, inputs) histories ending at the given rows: ar_dl_model.window_rows, on the GPU.
    idx = rows[:, None] + OFFSETS[None, :]
    return X_dev[torch.maximum(idx, ROW_START[rows][:, None])]

# Answers: failure within 30 days, within 7 days, and which part (only known for rows before a failure).
comp_rows = (fit_mask | valid_mask) & df["failure_next_30d"].to_numpy() & df["next_failure_component"].notna().to_numpy()
COMPONENTS = sorted(df.loc[comp_rows, "next_failure_component"].unique())
part = df["next_failure_component"].map({c: i for i, c in enumerate(COMPONENTS)}).fillna(-1).to_numpy(np.int64)
part[~df["failure_next_30d"].to_numpy()] = -1
Y30 = torch.from_numpy(y_all.astype(np.float32)).to(DEVICE)
Y7 = torch.from_numpy(df["failure_next_7d"].to_numpy(np.float32)).to(DEVICE)
YC = torch.from_numpy(part).to(DEVICE)
print(f"{len(FEATURES)} inputs x {WINDOW_STEPS} steps ({WINDOW_STEPS * STEP_HOURS / 24:.0f} days) per window | "
      f"{len(LOG_COLS)} inputs log-compressed | {X_dev.numel() * 4 / 1e9:.2f} GB on {DEVICE} | "
      f"parts: {COMPONENTS} | {time.time() - t0:.0f} s")

In [ ]:
DAYS_AGO = (WINDOW_STEPS - 1 - np.arange(WINDOW_STEPS)) * STEP_HOURS / 24

def day_axis(ax):
    # Label a window's step axis with days before the prediction.
    ticks = np.arange(WINDOW_STEPS - 1, -1, -max(1, 48 // STEP_HOURS))[::-1]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"-{d:.0f} d" if d else "now" for d in DAYS_AGO[ticks]])

cand = np.flatnonzero(fit_mask & (y_all == 1) & df["days_to_next_failure"].between(1, 3).to_numpy())
if len(cand):
    r = int(cand[len(cand) // 2])
    w = windows(torch.as_tensor([r], device=DEVICE))[0].cpu().numpy()
    top = np.argsort(w.std(axis=0))[::-1][:15]
    fig, ax = plt.subplots(figsize=(15, 5))
    im = ax.imshow(w[:, top].T, aspect="auto", cmap="coolwarm", vmin=-3, vmax=3)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([FEATURES[i] for i in top])
    day_axis(ax)
    plt.colorbar(im, label="scaled value (0 = fit-month average)")
    ax.set_title(f"One window as the network sees it: {df.at[r, 'asset_id']}, "
                 f"{df.at[r, 'days_to_next_failure']:.1f} days before a {df.at[r, 'next_failure_component']} failure")
    plt.tight_layout()
    plt.show()

## 6. Three networks
Three common ways to read a sequence are tried:

| network | how it reads the 14 days |
|---|---|
| **GRU** (gated recurrent unit) | Step by step, with a running memory, like an engineer scrolling through the log. A short convolution first smooths hour-to-hour noise. |
| **TCN** (temporal convolutional network) | Slides pattern detectors along the sequence. Each layer looks twice as far back as the one before. Fast, and good at sudden changes. |
| **Transformer** | Every step can look directly at every other step (self-attention), for example to compare today with 10 days ago. |

All three end the same way:
* The **latest step** and an **attention-weighted summary** of all steps are combined.
* Three output "heads" read the result: failure within 30 days, failure within 7 days, and which part.

Learning the 7-day and "which part" questions as well helps the network focus on what really comes before a failure.
The code is in `ar_dl_model.py` (written in section 5).

In [ ]:
# ARNet comes from ar_dl_model.py (section 5), the same code the dashboard uses.
for arch in ARCHS:
    n = sum(p.numel() for p in ARNet(arch, len(FEATURES), len(COMPONENTS), HIDDEN, WINDOW_STEPS).parameters())
    print(f"{arch:<12} {n:>9,} weights")

## 7. Train and pick the best network
* Each epoch (training round) draws a fresh sample of windows from the fit months.
* `POS_SHARE` of the windows end in the 30 days before a failure. Failures are rare, so without this the network
  would hardly ever see one. Section 8 corrects the probabilities afterwards.
* The loss is: 30-day failure + ½ × 7-day failure + 0.3 × "which part" (counted only for windows before a failure).
* After each epoch, the network is scored on validation windows (**PR-AUC**). The best epoch is kept.
* Training stops when the score has not improved for `PATIENCE` epochs.
* Every network in `ARCHS` is trained the same way. The one with the best validation PR-AUC is kept.

In [ ]:
FIT_POS = np.flatnonzero(fit_mask & (y_all == 1))
FIT_NEG = np.flatnonzero(fit_mask & (y_all == 0))
VALID_ROWS, TEST_ROWS = np.flatnonzero(valid_mask), np.flatnonzero(test_mask)
MONITOR = np.sort(np.random.default_rng(SEED).choice(VALID_ROWS, min(len(VALID_ROWS), 3_000 if SMOKE else 60_000),
                                                     replace=False))

def sample_rows(rng):
    n_pos = int(SAMPLES_PER_EPOCH * POS_SHARE)
    rows = np.concatenate([rng.choice(FIT_POS, n_pos), rng.choice(FIT_NEG, SAMPLES_PER_EPOCH - n_pos)])
    return torch.from_numpy(rng.permutation(rows)).to(DEVICE)

def loss_fn(out, rows):
    o30, o7, opart = out
    loss = (F.binary_cross_entropy_with_logits(o30.float(), Y30[rows])
            + 0.5 * F.binary_cross_entropy_with_logits(o7.float(), Y7[rows]))
    yc = YC[rows]
    known = yc >= 0
    if known.any():
        loss = loss + 0.3 * F.cross_entropy(opart[known].float(), yc[known])
    return loss

@torch.no_grad()
def predict(net, rows):
    # Raw 30-day failure probability and part probabilities for the windows that end at `rows`.
    net.eval()
    batch = 512 if SMOKE else 4096
    p30, parts = [], []
    for i in range(0, len(rows), batch):
        r = torch.as_tensor(rows[i:i + batch], device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=AMP):
            o30, _, opart = net(windows(r))
        p30.append(torch.sigmoid(o30.float()).cpu())
        parts.append(torch.softmax(opart.float(), dim=1).cpu())
    return torch.cat(p30).numpy(), torch.cat(parts).numpy()

def train(arch):
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    net = ARNet(arch, len(FEATURES), len(COMPONENTS), HIDDEN, WINDOW_STEPS).to(DEVICE)
    steps = SAMPLES_PER_EPOCH // BATCH
    opt = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LEARNING_RATE, total_steps=EPOCHS * steps)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    best, best_state, history, waited, t0 = -1.0, None, [], 0, time.time()
    for epoch in range(1, EPOCHS + 1):
        net.train()
        rows, total = sample_rows(rng), 0.0
        for b in range(steps):
            r = rows[b * BATCH:(b + 1) * BATCH]
            with torch.autocast(DEVICE.type, enabled=AMP):
                out = net(windows(r))
            loss = loss_fn(out, r)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            total += loss.detach()
        score = safe_ap(y_all[MONITOR], predict(net, MONITOR)[0])
        score = 0.0 if np.isnan(score) else score
        history.append({"arch": arch, "epoch": epoch, "train_loss": float(total) / steps,
                        "valid_pr_auc": score, "minutes": (time.time() - t0) / 60})
        print(f"  {arch:<11} epoch {epoch:>2}  loss {history[-1]['train_loss']:.4f}  "
              f"validation PR-AUC {score:.3f}  ({history[-1]['minutes']:.1f} min)")
        if score > best:
            best, waited = score, 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            waited += 1
            if waited >= PATIENCE:
                break
    net.load_state_dict(best_state)
    return net, history

runs, history = {}, []
for arch in ARCHS:
    net, hist = train(arch)
    h = pd.DataFrame(hist)
    best_row = h.loc[h["valid_pr_auc"].idxmax()]
    runs[arch] = {"net": net, "weights": sum(p.numel() for p in net.parameters()), "epochs run": len(h),
                  "best epoch": int(best_row["epoch"]), "best validation PR-AUC": round(float(best_row["valid_pr_auc"]), 3),
                  "minutes": round(float(h["minutes"].iloc[-1]), 1)}
    history += hist
selection = pd.DataFrame([{"network": a, **{k: v for k, v in r.items() if k != "net"}} for a, r in runs.items()])
selection = selection.set_index("network")
BEST_ARCH = selection["best validation PR-AUC"].idxmax()
model = runs[BEST_ARCH]["net"]
for r in runs.values():
    r["net"] = None
if AMP:
    torch.cuda.empty_cache()
display(selection)
print(f"kept: {BEST_ARCH} (random guess scores {y_all[MONITOR].mean():.3f})")

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for arch, h in hist.groupby("arch"):
    axes[0].plot(h["epoch"], h["train_loss"], "o-", label=arch)
    axes[1].plot(h["epoch"], h["valid_pr_auc"], "o-", label=arch)
axes[1].axhline(y_all[MONITOR].mean(), color="grey", ls=":", label="random guess")
axes[0].set(title="Training loss (lower = better)", xlabel="epoch")
axes[1].set(title="Validation PR-AUC (higher = better)", xlabel="epoch")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()
raw_val, _ = predict(model, VALID_ROWS)
y_val = y_all[valid_mask]
print(f"{BEST_ARCH}: PR-AUC on all validation rows {safe_ap(y_val, raw_val):.3f}")

## 8. Realistic probabilities (calibration)
Training used 30 % failure windows, far more than in reality (about 8 %), so the raw network scores are too high.
**Isotonic calibration** on the validation months maps them to probabilities that match how often failures
really followed. This is the same code as in the LightGBM notebook.

In [ ]:
iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(raw_val, y_val)
CAL_X, CAL_Y = [float(v) for v in iso.X_thresholds_], [float(v) for v in iso.y_thresholds_]

def calibrate(raw):
    return np.clip(np.interp(raw, CAL_X, CAL_Y), 0.0, 1.0)

p_val = calibrate(raw_val)
print(f"{len(CAL_X)} calibration points | validation: mean predicted {p_val.mean():.3f} vs observed {y_val.mean():.3f}")

## 9. Alarm level (threshold)
The same rule as the LightGBM notebook. On the validation months, it picks the level that catches the most
failures while staying within `FALSE_ALARM_BUDGET` false-alarm days per recloser per month. If no level fits the
budget, the level with the best F1 is used.

In [ ]:
def event_metrics(frame, warn, fails):
    warn = pd.Series(np.asarray(warn, dtype=bool), index=frame.index)
    by_asset = {a: g for a, g in frame.assign(_w=warn).groupby("asset_id")}
    rows = []
    for fl in fails.itertuples():
        g = by_asset.get(fl.asset_id)
        hit = pd.Series(dtype="datetime64[ns]") if g is None else g.loc[
            g["_w"] & (g["ts"] < fl.failure_time) & (g["ts"] > fl.failure_time - pd.Timedelta(days=HORIZON_DAYS)), "ts"]
        lead = (fl.failure_time - hit.min()).total_seconds() / 86400 if len(hit) else np.nan
        rows.append((fl.asset_id, fl.failure_time, fl.component, len(hit) > 0, lead))
    per = pd.DataFrame(rows, columns=["asset_id", "failure_time", "component", "caught", "warning_days"])
    alarms = frame[warn.to_numpy() & ~frame["failure_next_30d"].to_numpy()]
    false_days = alarms.assign(day=alarms["ts"].dt.floor("D")).drop_duplicates(["asset_id", "day"]).shape[0]
    ar_months = frame.groupby("asset_id")["ts"].agg(lambda s: (s.max() - s.min()).days / 30.44).sum()
    return per, false_days / max(ar_months, 1e-9)

val_frame = df.loc[valid_mask, ["asset_id", "ts", "failure_next_30d"]].assign(p=p_val)
val_fails = failures_between(VALID_START, TEST_START)
grid = []
for thr in np.unique(np.quantile(p_val, np.linspace(0.5, 0.999, 80))):
    per, fa = event_metrics(val_frame, val_frame["p"] >= thr, val_fails)
    grid.append((float(thr), per["caught"].mean() if len(per) else np.nan, fa))
grid = pd.DataFrame(grid, columns=["threshold", "share_of_failures_caught", "false_alarm_days_per_ar_month"])
ok = grid[(grid["false_alarm_days_per_ar_month"] <= FALSE_ALARM_BUDGET) & grid["share_of_failures_caught"].notna()]
if len(ok) and ok["share_of_failures_caught"].max() > 0:
    THRESHOLD = float(ok.sort_values(["share_of_failures_caught", "threshold"], ascending=[False, False]).iloc[0]["threshold"])
    how = f"most failures caught within {FALSE_ALARM_BUDGET} false-alarm day(s) per AR-month"
else:
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-9)
    THRESHOLD = float(thr[np.nanargmax(f1[:-1])]) if len(thr) else 0.5
    how = "best F1 (no level met the false-alarm budget)"
print(f"alarm level: probability >= {THRESHOLD:.3f} ({how})")

## 10. Honest test: deep learning vs LightGBM vs Health Index
All numbers below come from the **test months**, which no model saw.
* **LightGBM** appears when `ar_ml_model.zip` is found in Drive/AR_ML, or when the notebook runs inside the
  project, where it uses `models/`.
* **ROC-AUC / PR-AUC:** how well risky rows are ranked above safe ones. PR-AUC is the stricter one for rare events.
* **Failures caught:** a warning appeared in the 30 days before the failure.
* **Warning days:** how early the first warning came.
* **"DL or …":** warns when either one does. It shows what you gain by using both.

In [ ]:
test = df.loc[test_mask, ["asset_id", "ts", "health_index", "failure_next_30d", "next_failure_component"]].copy()
raw_test, part_test = predict(model, TEST_ROWS)
test["p"] = calibrate(raw_test)
test_fails = failures_between(TEST_START, t_last + GAP)

def load_lgbm():
    for z in (LGBM_ZIP, f"/content/drive/MyDrive/{DRIVE_FOLDER}/{LGBM_ZIP}", f"/content/{LGBM_ZIP}"):
        if os.path.exists(z):
            with zipfile.ZipFile(z) as fh:
                fh.extractall("lgbm_bundle")
            break
    folder = next((d for d in ("lgbm_bundle", os.path.join("..", "models"))
                   if os.path.exists(os.path.join(d, "ml_meta.json"))
                   and os.path.exists(os.path.join(d, "model_failure30d.txt"))), None)
    if folder is None:
        print(f"No LightGBM bundle found: put {LGBM_ZIP} in Drive/{DRIVE_FOLDER} to add it to the comparison.")
        return None
    with open(os.path.join(folder, "ml_meta.json")) as fh:
        meta = json.load(fh)
    if meta.get("feature_version") != FEATURE_VERSION or not set(meta["features"]) <= set(FEATURES):
        print("The LightGBM bundle uses another feature recipe; retrain it with AR_ML_Colab.ipynb. Skipped.")
        return None
    if str(meta.get("data_until_utc")) != str(df["ts"].max()):
        print(f"NOTE: the LightGBM bundle was trained on data up to {meta.get('data_until_utc')}, this file ends "
              f"{df['ts'].max()}. The comparison is only fair when both notebooks used the same data file.")
    comp_path = os.path.join(folder, "model_component.txt")
    comp = lgb.Booster(model_file=comp_path) if os.path.exists(comp_path) and meta.get("component_classes") else None
    print(f"LightGBM bundle {meta.get('model_version')} from {folder}")
    return {"booster": lgb.Booster(model_file=os.path.join(folder, "model_failure30d.txt")),
            "meta": meta, "component": comp}

LGBM = load_lgbm()
if LGBM:
    X_test_lgbm = X_all.loc[test_mask, LGBM["meta"]["features"]].to_numpy(np.float32)
    cal = LGBM["meta"]["calibration"]
    test["p_lgbm"] = np.clip(np.interp(LGBM["booster"].predict(X_test_lgbm), cal["x"], cal["y"]), 0.0, 1.0)
    LGBM_THRESHOLD = float(LGBM["meta"]["threshold"])

DL = f"Deep learning ({BEST_ARCH})"
HI = "Health Index (HI < 60)"
dl_warn, hi_warn = test["p"] >= THRESHOLD, test["health_index"] < HI_HEALTHY_MIN
dl_rel = test["p"] / max(THRESHOLD, 1e-9)                       # >= 1 means the network warns
hi_rel = (100 - test["health_index"]) / (100 - HI_HEALTHY_MIN)  # > 1 means the HI warns
systems = {DL: (test["p"], dl_warn)}
if LGBM:
    lgbm_warn = test["p_lgbm"] >= LGBM_THRESHOLD
    systems["LightGBM"] = (test["p_lgbm"], lgbm_warn)
systems[HI] = (100 - test["health_index"], hi_warn)
if LGBM:
    systems["DL or LightGBM"] = (np.maximum(dl_rel, test["p_lgbm"] / max(LGBM_THRESHOLD, 1e-9)), dl_warn | lgbm_warn)
systems["DL or HI"] = (np.maximum(dl_rel, hi_rel), dl_warn | hi_warn)

results, per_failure = {}, {}
for name, (score, warn) in systems.items():
    per, fa = event_metrics(test, warn, test_fails)
    per_failure[name] = per
    results[name] = {"ROC-AUC": round(safe_auc(test["failure_next_30d"], score), 3),
                     "PR-AUC": round(safe_ap(test["failure_next_30d"], score), 3),
                     "failures caught": f"{int(per['caught'].sum())} / {len(per)}",
                     "median warning days": round(per["warning_days"].median(), 1) if len(per) else np.nan,
                     "false-alarm days per AR-month": round(fa, 2)}
comparison = pd.DataFrame(results).T
print(f"random-guess PR-AUC = {test['failure_next_30d'].mean():.3f} | "
      f"Brier score of the network (lower = better) = {brier_score_loss(test['failure_next_30d'], test['p']):.4f}")
display(comparison)
by_comp = pd.DataFrame({name: per.groupby("component")["caught"].agg(lambda s: f"{int(s.sum())} / {len(s)}")
                        for name, per in per_failure.items()})
by_comp.index.name = "failures caught by component"
display(by_comp)

In [ ]:
y_test = test["failure_next_30d"].to_numpy()
probs = [(DL, "p", "#e67e22")] + ([("LightGBM", "p_lgbm", "#00a3cc")] if LGBM else [])
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if 0 < y_test.sum() < len(y_test):
    for name, score in [(n, test[c]) for n, c, _ in probs] + [("Health Index", 100 - test["health_index"])]:
        pr, rc, _ = precision_recall_curve(y_test, score)
        axes[0].plot(rc, pr, label=f"{name} (PR-AUC {safe_ap(y_test, score):.2f})")
axes[0].axhline(y_test.mean(), color="grey", ls=":", label="random guess")
axes[0].set(xlabel="recall", ylabel="precision", title="Precision vs recall (test)")
axes[0].legend()
for name, _, color in probs:
    axes[1].hist(per_failure[name]["warning_days"].dropna(), bins=15, alpha=0.6, color=color, label=name)
axes[1].set(title="Warning days before caught failures", xlabel="days")
axes[1].legend()
if y_test.sum() > 0:
    for name, col, color in probs:
        frac, mean_p = calibration_curve(y_test, test[col], n_bins=10, strategy="quantile")
        axes[2].plot(mean_p, frac, "o-", color=color, label=name)
axes[2].plot([0, 1], [0, 1], ":", color="grey", label="perfect")
axes[2].set(title="Calibration (test)", xlabel="predicted probability", ylabel="observed frequency")
axes[2].legend()
plt.tight_layout()
plt.show()

## 11. One recloser over time
The HI (blue, left axis) and the failure probabilities (right axis) for the test recloser where the network and
the HI together caught the most failures. Black lines are recorded failures; dashed lines are the alarm levels.

In [ ]:
if len(test_fails):
    both = per_failure[DL].groupby("asset_id")["caught"].sum() + per_failure[HI].groupby("asset_id")["caught"].sum()
    aid = both.sort_values(kind="stable").index[-1]
    cols = ["health_index", "p"] + (["p_lgbm"] if LGBM else [])
    a = test[test["asset_id"] == aid].set_index("ts")[cols].resample("1h").asfreq()
    fig, ax1 = plt.subplots(figsize=(15, 4))
    ax1.plot(a.index, a["health_index"], color="tab:blue", lw=1)
    ax1.axhline(HI_HEALTHY_MIN, color="tab:blue", ls="--", lw=0.8)
    ax1.set_ylim(0, 102)
    ax1.set_ylabel("Health Index", color="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(a.index, a["p"], color="tab:red", lw=1, label=DL)
    ax2.axhline(THRESHOLD, color="tab:red", ls="--", lw=0.8)
    if LGBM:
        ax2.plot(a.index, a["p_lgbm"], color="tab:orange", lw=0.8, alpha=0.8, label="LightGBM")
    ax2.set_ylim(0, 1.02)
    ax2.set_ylabel("failure probability (30 days)", color="tab:red")
    ax2.legend(loc="upper left")
    for k, fl in enumerate(test_fails[test_fails["asset_id"] == aid].itertuples()):
        ax1.axvline(fl.failure_time, color="black", lw=1.5)
        ax1.text(fl.failure_time, 3 + 9 * (k % 2), f" {fl.component}", fontsize=9)
    ax1.set_title(f"{aid}: Health Index vs failure probability (test months)")
    plt.show()

## 12. Which part is likely to fail?
The network's third head names the most likely part. It is checked on the test rows in the 30 days before a
failure, and compared with LightGBM's separate component model when that is available.

In [ ]:
pos = y_test & test["next_failure_component"].isin(COMPONENTS).to_numpy()
if len(COMPONENTS) >= 2 and pos.any():
    predicted = np.array(COMPONENTS)[part_test[pos].argmax(axis=1)]
    actual = test.loc[pos, "next_failure_component"].to_numpy()
    print(f"{DL}: right part on {(predicted == actual).mean():.0%} of test rows before failures")
    if LGBM and LGBM["component"] is not None:
        lgbm_parts = np.array(LGBM["meta"]["component_classes"])[LGBM["component"].predict(X_test_lgbm[pos]).argmax(axis=1)]
        print(f"LightGBM component model: right part on {(lgbm_parts == actual).mean():.0%} of the same rows")
    display(pd.crosstab(pd.Series(actual, name="actual"), pd.Series(predicted, name="predicted (deep learning)")))
else:
    print("Not enough different failing parts in the training months to name the part.")

## 13. What does the network look at?
Neural networks have no built-in SHAP values like LightGBM, so **gradients** are used. They measure how much each
input, at each step, pushes the 30-day risk (log-odds) up or down. The reference is an "average recloser": all
inputs at their fit-month average.
* **Left:** the inputs that move the risk most, averaged over test windows (gradient × input).
* **Right:** the same, per step. It shows how far back in the 14 days the network looks.
* **Next chart:** **integrated gradients** for the first warning before one caught failure. Red pushed the risk
  up, blue pushed it down.

In [ ]:
def grad_x_input(net, x):
    net.eval()
    x = x.detach().clone().requires_grad_(True)
    with torch.backends.cudnn.flags(enabled=False):             # cuDNN cannot back-propagate a GRU in eval mode
        net(x)[0].sum().backward()
    return (x.grad * x).detach()

def integrated_gradients(net, x, n=32):
    # For one window x (1 x steps x inputs), starting from the all-average window.
    net.eval()
    path = (torch.linspace(1 / n, 1, n, device=x.device)[:, None, None] * x).requires_grad_(True)
    with torch.backends.cudnn.flags(enabled=False):
        net(path)[0].sum().backward()
    return (path.grad.mean(dim=0) * x[0]).detach().cpu().numpy()

sample = np.random.default_rng(SEED).choice(TEST_ROWS, min(len(TEST_ROWS), 256 if SMOKE else 4096), replace=False)
total = 0
for i in range(0, len(sample), 256):
    total = total + grad_x_input(model, windows(torch.as_tensor(sample[i:i + 256], device=DEVICE))).abs().sum(dim=0)
total = (total / len(sample)).cpu().numpy()                     # steps x inputs
fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={"width_ratios": [1, 1.3]})
by_input = pd.Series(total.sum(axis=0), index=FEATURES).sort_values(ascending=False).head(20)
by_input[::-1].plot.barh(ax=axes[0], color="#00a3cc", title="Top 20 inputs (mean |gradient x input|)")
axes[1].plot(np.arange(WINDOW_STEPS), total.sum(axis=1), color="#e67e22")
day_axis(axes[1])
axes[1].set(title="How far back the network looks", ylabel="mean |gradient x input|")
plt.tight_layout()
plt.show()

In [ ]:
caught = per_failure[DL][per_failure[DL]["caught"]]
if len(caught):
    ex = caught.iloc[0]
    g = test[(test["asset_id"] == ex.asset_id) & (test["p"] >= THRESHOLD) & (test["ts"] < ex.failure_time)
             & (test["ts"] > ex.failure_time - pd.Timedelta(days=HORIZON_DAYS))]
    row = int(g.index[0])
    ig = integrated_gradients(model, windows(torch.as_tensor([row], device=DEVICE)))
    top = np.argsort(np.abs(ig).sum(axis=0))[::-1][:12]
    lim = np.abs(ig[:, top]).max() or 1.0
    fig, ax = plt.subplots(figsize=(15, 5))
    im = ax.imshow(ig[:, top].T, aspect="auto", cmap="coolwarm", vmin=-lim, vmax=lim)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([FEATURES[i] for i in top])
    day_axis(ax)
    plt.colorbar(im, label="push on the risk (red = up)")
    ax.set_title(f"{ex.asset_id}: first warning {test.loc[row, 'ts']:%Y-%m-%d %H:%M} before a {ex.component} "
                 f"failure on {ex.failure_time:%Y-%m-%d}")
    plt.tight_layout()
    plt.show()

## 14. Daily view: HI vs deep learning per recloser per day
The same daily comparison the project stores in `ar_daily_summary`, here with the network's probability:
* HI healthy = daily average HI ≥ 60;
* ML healthy = probability below the alarm level all day.

In [ ]:
d = test.assign(day=test["ts"].dt.tz_localize("UTC").dt.tz_convert("Asia/Colombo").dt.date)
daily = d.groupby(["asset_id", "day"]).agg(avg_hi=("health_index", "mean"), max_p=("p", "max")).reset_index()
daily["hi_healthy"] = daily["avg_hi"] >= HI_HEALTHY_MIN
daily["ml_healthy"] = daily["max_p"] < THRESHOLD
daily["agree"] = daily["hi_healthy"] == daily["ml_healthy"]
print(f"{len(daily):,} recloser-days | HI and ML agree on {daily['agree'].mean():.1%}")
display(pd.crosstab(daily["hi_healthy"].map({True: "HI: healthy", False: "HI: NOT healthy"}),
                    daily["ml_healthy"].map({True: "ML: healthy", False: "ML: NOT healthy"})))
top = daily.groupby("asset_id")["max_p"].max().sort_values(ascending=False).head(20).index
heat = daily[daily["asset_id"].isin(top)].pivot(index="asset_id", columns="day", values="max_p")
plt.figure(figsize=(16, 6))
plt.imshow(heat.to_numpy(dtype=float), aspect="auto", cmap="inferno", vmin=0, vmax=max(THRESHOLD * 2, 0.05))
plt.yticks(range(len(heat)), heat.index)
plt.colorbar(label="daily max ML probability")
plt.title("Daily ML risk for the 20 highest-risk reclosers (test months)")
plt.show()

## 15. Export the network
This writes `ar_dl_model.zip` with:
* `model_dl.pt`: the network weights (PyTorch) and its size settings;
* `dl_meta.json`: everything needed to rebuild the inputs and read the output. That is the feature list, scaling,
  window length, alarm level, calibration, part names and the test results.

**Install it on the PC:** unzip it into the project's **`models/dl/`** folder (not `models/`, which holds the
LightGBM model), then run `python main.py dl-score`. The dashboard's **Deep Learning** tab then shows the network's
risk next to LightGBM and the HI, and the collector scores every 15 minutes. A copy is saved in Drive/AR_ML when
Drive is connected.

In [ ]:
os.makedirs("export_dl", exist_ok=True)
for old in os.listdir("export_dl"):
    os.remove(os.path.join("export_dl", old))
NET_CONFIG = {"arch": BEST_ARCH, "n_inputs": len(FEATURES), "n_components": len(COMPONENTS),
              "width": HIDDEN, "steps": WINDOW_STEPS}
torch.save({"config": NET_CONFIG, "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}},
           "export_dl/model_dl.pt")
dl_res = results[DL]
metrics = {"test_roc_auc": dl_res["ROC-AUC"], "test_pr_auc": dl_res["PR-AUC"],
           "test_failures_caught": dl_res["failures caught"],
           "test_median_warning_days": dl_res["median warning days"],
           "test_false_alarm_days_per_ar_month": dl_res["false-alarm days per AR-month"],
           "hi_failures_caught": results[HI]["failures caught"],
           "dl_or_hi_failures_caught": results["DL or HI"]["failures caught"],
           "validation_pr_auc": float(selection.loc[BEST_ARCH, "best validation PR-AUC"])}
if LGBM:
    metrics["lightgbm_failures_caught"] = results["LightGBM"]["failures caught"]
    metrics["dl_or_lightgbm_failures_caught"] = results["DL or LightGBM"]["failures caught"]
now = dt.datetime.now(dt.timezone.utc)
meta = {"model_type": "pytorch-sequence", "net_version": NET_VERSION, "arch": BEST_ARCH, "net_config": NET_CONFIG,
        "feature_version": FEATURE_VERSION, "features": FEATURES,
        "window_steps": WINDOW_STEPS, "step_hours": STEP_HOURS,
        "scaling": {"log1p": LOG_COLS, "mean": MU.tolist(), "std": SD.tolist(), "clip": CLIP},
        "threshold": THRESHOLD, "calibration": {"x": CAL_X, "y": CAL_Y},
        "component_classes": COMPONENTS, "metrics": metrics,
        "selection": selection.reset_index().to_dict("records"), "horizon_days": HORIZON_DAYS,
        "trained_utc": now.strftime("%Y-%m-%dT%H:%M:%S"), "data_until_utc": str(df["ts"].max()),
        "torch_version": torch.__version__, "model_version": f"dl-{BEST_ARCH}-" + now.strftime("%Y%m%d-%H%M")}
with open("export_dl/dl_meta.json", "w") as fh:
    json.dump(meta, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
with zipfile.ZipFile("ar_dl_model.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir("export_dl")):
        z.write(os.path.join("export_dl", fn), fn)
print("ar_dl_model.zip:", sorted(os.listdir("export_dl")), "| model version", meta["model_version"])
display(pd.Series(metrics).to_frame("test result"))
drive_dir = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
if IN_COLAB and not SMOKE:
    if os.path.isdir(drive_dir):
        shutil.copy("ar_dl_model.zip", drive_dir)
        print("copy saved in Drive:", drive_dir)
    from google.colab import files
    files.download("ar_dl_model.zip")

In [ ]:
# Check: rebuild the network from the exported files alone and score a few test rows from the raw inputs,
# with the functions the dashboard uses (ar_dl_model.window_rows and scale_inputs).
saved = torch.load("export_dl/model_dl.pt", map_location="cpu", weights_only=True)
with open("export_dl/dl_meta.json") as fh:
    m = json.load(fh)
check = ARNet(**saved["config"]).eval()
check.load_state_dict(saved["state_dict"])
s = m["scaling"]
log_idx = [m["features"].index(c) for c in s["log1p"]]
rows = TEST_ROWS[-5:]
win = window_rows(rows, ROW_START_NP[rows], m["window_steps"], m["step_hours"])     # hourly rows
x = np.stack([scale_inputs(X_all.iloc[w][m["features"]].to_numpy(np.float32), log_idx, s["mean"], s["std"],
                           s["clip"]) for w in win])
with torch.no_grad():
    raw_check = torch.sigmoid(check(torch.from_numpy(x))[0]).numpy()
print("raw scores from the bundle:", np.round(raw_check, 4), "| in the notebook:", np.round(raw_test[-5:], 4))
assert np.allclose(raw_check, raw_test[-5:], atol=0.01), "the exported bundle does not reproduce the notebook"
print("probabilities:", np.round(np.interp(raw_check, m["calibration"]["x"], m["calibration"]["y"]), 4))

### Reading the results honestly
* **The data is simulated.** The network learns the patterns the simulator built in. Test it on real NOJA data
  before relying on it.
* **Deep learning is not automatically better.** On table data like this, gradient-boosted trees (LightGBM) are
  often as good or better. Judge only by the section 10 test table, not by the validation scores.
* **Different strengths.** If the network and LightGBM catch different failures, "DL or LightGBM" shows the gain
  of using both.
* **Results vary a little between runs** on a GPU. For a firm conclusion, run the notebook twice and compare.
* **Retrain** when more data is available: generate a new file on the PC, then run both notebooks again on the same file.
  Replace the files in `models/dl/` and run `python main.py dl-score`.